In [1]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [6]:
# Load the local CSV files
users = pd.read_csv('users.csv')
feedback = pd.read_csv('match_feedback.csv')

print(f"Loaded {len(users)} users and {len(feedback)} feedback records.")

Loaded 50 users and 50 feedback records.


In [7]:
# Combine the two mandatory text columns into one for analysis
users['combined_bio'] = users['professional_summary'] + " " + users['about_me']

# Convert text into numbers using TF-IDF
tfidf = TfidfVectorizer(stop_words='english')
matrix = tfidf.fit_transform(users['combined_bio'])

In [8]:
# Calculate the similarity matrix (how much every user matches every other user)
text_sim_matrix = cosine_similarity(matrix)

print("NLP processing complete. Text similarity matrix generated.")

NLP processing complete. Text similarity matrix generated.


In [9]:
# Rule dictionary for MBTI compatibility (Score 0.0 to 1.0)
mbti_rules = {
    ('INTJ', 'ENFP'): 1.0, ('INTP', 'INFJ'): 0.9,
    ('ENTJ', 'INTP'): 0.9, ('ENFP', 'INFJ'): 1.0,
    ('ISTJ', 'ESFJ'): 0.8, ('ENFJ', 'INFP'): 0.9
}

def calculate_mbti_score(type1, type2):
    # Check both directions in the dictionary
    if (type1, type2) in mbti_rules:
        return mbti_rules[(type1, type2)]
    if (type2, type1) in mbti_rules:
        return mbti_rules[(type2, type1)]
    return 0.5  # Neutral score if no specific rule exists

def calculate_location_score(loc1, loc2):
    return 1.0 if loc1 == loc2 else 0.0

In [10]:
def get_top_matches(target_id, users_df, sim_data, weights):
    # Get index of the target user
    idx = users_df.index[users_df['user_id'] == target_id]
    user_data = users_df.iloc[idx]

    all_scores = []

    for i, other_user in users_df.iterrows():
        if other_user['user_id'] == target_id:
            continue

        # Get individual component scores
        text_score = sim_data[idx][i]
        mbti_score = calculate_mbti_score(user_data['mbti'], other_user['mbti'])
        loc_score = calculate_location_score(user_data['location'], other_user['location'])

        # Apply the weighted formula
        final_score = (weights['w1'] * text_score) + \
                      (weights['w2'] * mbti_score) + \
                      (weights['w3'] * loc_score)

        all_scores.append({
            'ID': other_user['user_id'],
            'Name': other_user['name'],
            'Match %': round(final_score * 100, 2)
        })

    # Sort by highest match percentage
    return sorted(all_scores, key=lambda x: x['Match %'], reverse=True)


In [11]:
def tune_weights(user_id, feedback_df):
    # Default weights: Text=50%, MBTI=30%, Location=20%
    w = {'w1': 0.5, 'w2': 0.3, 'w3': 0.2}

    # Filter feedback for this specific user
    user_feedback = feedback_df[feedback_df['user_id'] == user_id]

    # Simple logic: If user rejects matches (0), reduce MBTI influence
    rejections = len(user_feedback[user_feedback['action'] == 0])
    if rejections > 3:
        w['w1'] += 0.1  # Trust text more
        w['w2'] -= 0.1  # Trust MBTI less

    return w

In [14]:
# This function calculates how well other users match a specific 'target_id'

def get_top_matches(target_id, users_df, sim_data, weights):
    # Step 1: Find the target user's details first
    target_user_index = -1
    for idx in range(len(users_df)):
        if users_df.iloc[idx]['user_id'] == target_id:
            target_user_index = idx
            break

    # Get target user data (MBTI and Location) for comparison
    target_mbti = users_df.iloc[target_user_index]['mbti']
    target_location = users_df.iloc[target_user_index]['city']

    # Step 2: Create a list to store all the scores
    all_user_scores = []

    # Step 3: Loop through every user in the dataset
    for i in range(len(users_df)):
        other_user = users_df.iloc[i]

        # Don't match the user with themselves
        if other_user['user_id'] == target_id:
            continue

        # --- APPLYING THE CORE FORMULA [1] ---
        # 1. Get Text Similarity from the NLP Matrix
        text_sim = sim_data[target_user_index][i]

        # 2. Get MBTI Match score from our rules
        other_mbti = other_user['mbti']
        mbti_score = calculate_mbti_score(target_mbti, other_mbti)

        # 3. Get Location score (1 if same city, 0 if different)
        other_location = other_user['city']
        if target_location == other_location:
            loc_score = 1.0

        else:
            loc_score = 0.0

        # FINAL FORMULA: (w1 * Text) + (w2 * MBTI) + (w3 * Location)
        # Using weights tuned by the Feedback Loop [2]
        total_score = (weights['w1'] * text_sim) + \
                      (weights['w2'] * mbti_score) + \
                      (weights['w3'] * loc_score)

        # Save the result as a simple dictionary
        match_info = {
            'ID': other_user['user_id'],
            'Name': other_user['name'],
            'Score': round(total_score * 100, 2)
        }
        all_user_scores.append(match_info)

    # Step 4: Sort the list by the 'Score' field (Highest to Lowest)
    # Using a basic sorting key logic
    all_user_scores.sort(key=lambda x: x['Score'], reverse=True)

    return all_user_scores

# --- RUNNING THE PROGRAM ---

# 1. Define the user we want to find matches for
test_user_id = "U001"

# 2. Get personalized weights based on user history (Module 3: ML Layer) [2]
# This checks if the user prefers personality (MBTI) or career (Text)
current_weights = tune_weights(test_user_id, feedback)

# 3. Generate the list of matches
final_recommendations = get_top_matches(test_user_id, users, text_sim_matrix, current_weights)

# 4. Display the results in a beginner-friendly format
print("------------------------------------------")
print(f"TOP MATCHES FOR: {test_user_id}")
print(f"Active Weights: {current_weights}")
print("------------------------------------------")

# Show only the top 5 results
for i in range(5):
    match = final_recommendations[i]
    print(f"Rank {i+1}: {match['Name']} [{match['ID']}] -> {match['Score']}% Match")


------------------------------------------
TOP MATCHES FOR: U001
Active Weights: {'w1': 0.5, 'w2': 0.3, 'w3': 0.2}
------------------------------------------
Rank 1: Vivek Nair [U011] -> 85.0% Match
Rank 2: Sachin Patel [U021] -> 85.0% Match
Rank 3: Deepak Singh [U031] -> 85.0% Match
Rank 4: Rakesh Nanda [U041] -> 85.0% Match
Rank 5: Khushi Jain [U040] -> 62.94% Match
